# BayHunter output models: quick inspection and forward dispersion

This tutorial loads BayHunter chain outputs (e.g. `c000_p1models.npy`), shows their shapes,
plots a few model profiles, and compares forward-modeled dispersion curves to the
observed (true) dispersion curve.

Update the file paths in the next cell if you want to point to a different run.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from BayHunter.Models import Model
from BayHunter.data.model import SeismicModel

root = Path.cwd()
results_dir = Path("/home/schaetti/Projets/RECHERCHES/Recherches/MIGRATE/reps/BayHunterNS/tutorial/migrate/results/tempest_10m_00000/data")
observed_dir = Path("/home/schaetti/Projets/RECHERCHES/Recherches/MIGRATE/reps/BayHunterNS/tutorial/migrate/observed")

model_files = sorted(results_dir.glob("**/*_p1models.npy"))
if not model_files:
    raise FileNotFoundError("No *_p1models.npy files found under tutorial/migrate/results.")

vpvs_files = [p.with_name(p.name.replace("models", "vpvs")) for p in model_files]
true_dispersion_path = observed_dir / "tempest_H_0000.dat"

print(f"Found {len(model_files)} model files.")
print(f"Observed dispersion file: {true_dispersion_path}")


In [ ]:
models_list = []
vpvs_list = []

for model_path, vpvs_path in zip(model_files, vpvs_files):
    models_list.append(np.load(model_path))
    vpvs_list.append(np.load(vpvs_path))

models = np.vstack(models_list)
vpvs_values = np.concatenate(vpvs_list)

print("Stacked models array shape:", models.shape)
print("Stacked Vp/Vs array shape:", vpvs_values.shape)

valid_lengths = np.sum(~np.isnan(models), axis=1)
print("Valid parameter counts (min/mean/max):", valid_lengths.min(), valid_lengths.mean(), valid_lengths.max())


In [ ]:
true_model_path = observed_dir / "tempest_H_velmap_00000.dat"
true_model_data = np.loadtxt(true_model_path, skiprows=1)
true_model_depth = true_model_data[:, 0]
true_model_vs = true_model_data[:, 1]

print("True model points:", true_model_depth.size)


In [ ]:
def plot_model_profiles(models_array, vpvs_array, indices, true_depth=None, true_vs=None):
    fig, ax = plt.subplots(figsize=(6, 8))
    depth_max = None
    if true_depth is not None:
        depth_max = float(np.nanmax(true_depth))
    for idx in indices:
        model_row = models_array[idx]
        model_clean = model_row[~np.isnan(model_row)]
        if model_clean.size == 0:
            continue
        vpvs = float(vpvs_array[idx])
        _, vs_step, dep_step = Model.get_stepmodel(model_clean, vpvs=vpvs)
        if depth_max is not None:
            mask = dep_step <= depth_max
            dep_step = dep_step[mask]
            vs_step = vs_step[mask]
        ax.plot(vs_step, dep_step, label=f"model {idx}")
    if true_depth is not None and true_vs is not None:
        ax.plot(true_vs, true_depth, color="black", linewidth=2, label="true model")
        ax.set_ylim(depth_max, 0)
    else:
        ax.invert_yaxis()
    ax.set_xlabel("Vs (km/s)")
    ax.set_ylabel("Depth (km)")
    ax.set_title("Sampled BayHunter Models (clipped to true depth)")
    ax.grid(True)
    ax.legend()
    return fig, ax

sample_indices = np.linspace(0, len(models) - 1, 4, dtype=int)
plot_model_profiles(models, vpvs_values, sample_indices, true_model_depth, true_model_vs)
plt.show()


In [ ]:
true_data = np.loadtxt(true_dispersion_path, skiprows=1)
true_x = true_data[:, 0]
true_y = true_data[:, 1]

print("Observed dispersion points:", true_x.size)


In [ ]:
def forward_dispersion_for_models(models_array, vpvs_array, indices, periods):
    curves = []
    for idx in indices:
        model_row = models_array[idx]
        model_clean = model_row[~np.isnan(model_row)]
        if model_clean.size == 0:
            continue
        vpvs = float(vpvs_array[idx])
        seismic_model = SeismicModel(model_clean, vpvs)
        curve = seismic_model.forward(
            length=len(periods),
            min_p=float(periods.min()),
            max_p=float(periods.max()),
        )
        curves.append((idx, curve))
    return curves

curve_indices = np.linspace(0, len(models) - 1, 3, dtype=int)
synthetic_curves = forward_dispersion_for_models(models, vpvs_values, curve_indices, true_x)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(true_x, true_y, "k.", label="observed")
for idx, curve in synthetic_curves:
    ax.plot(curve.x, curve.y, label=f"synthetic {idx}")
ax.set_xlabel("Period (s)")
ax.set_ylabel("Group velocity (km/s)")
ax.set_title("Observed vs. Synthetic Dispersion Curves")
ax.grid(True)
ax.legend()
plt.show()


In [ ]:
def stack_stepmodels(models_array, vpvs_array, indices, depth_grid):
    stacked = []
    for idx in indices:
        model_row = models_array[idx]
        model_clean = model_row[~np.isnan(model_row)]
        if model_clean.size == 0:
            continue
        vpvs = float(vpvs_array[idx])
        _, vs_step, dep_step = Model.get_stepmodel(model_clean, vpvs=vpvs)
        stacked.append(np.interp(depth_grid, dep_step, vs_step))
    return np.asarray(stacked)

def summarize_curves(curves_array):
    summary = {
        "mean": np.nanmean(curves_array, axis=0),
        "median": np.nanmedian(curves_array, axis=0),
        "q1": np.nanpercentile(curves_array, 25, axis=0),
        "q3": np.nanpercentile(curves_array, 75, axis=0),
        "p5": np.nanpercentile(curves_array, 5, axis=0),
        "p95": np.nanpercentile(curves_array, 95, axis=0),
    }
    return summary


In [ ]:
def dispersion_misfit(observed_y, predicted_y):
    return float(np.mean((observed_y - predicted_y) ** 2))

best_curves = []
best_models = []
best_vpvs = []
misfits = []
all_curve_ys = []

for idx, model_row in enumerate(models):
    model_clean = model_row[~np.isnan(model_row)]
    if model_clean.size == 0:
        continue
    vpvs = float(vpvs_values[idx])
    seismic_model = SeismicModel(model_clean, vpvs)
    curve = seismic_model.forward(
        length=len(true_x),
        min_p=float(true_x.min()),
        max_p=float(true_x.max()),
    )
    all_curve_ys.append(curve.y)
    misfit = dispersion_misfit(true_y, curve.y)
    misfits.append((misfit, idx, curve.y))

misfits.sort(key=lambda item: item[0])
best_count = min(100, len(misfits))

for misfit, idx, curve_y in misfits[:best_count]:
    best_curves.append(curve_y)
    best_models.append(models[idx])
    best_vpvs.append(vpvs_values[idx])

best_curves = np.asarray(best_curves)
best_models = np.asarray(best_models)
best_vpvs = np.asarray(best_vpvs)
all_curve_ys = np.asarray(all_curve_ys)

disp_stats_best = summarize_curves(best_curves)
disp_stats_all = summarize_curves(all_curve_ys)

depth_grid = np.linspace(true_model_depth.min(), true_model_depth.max(), 200)
model_stack_best = stack_stepmodels(best_models, best_vpvs, np.arange(len(best_models)), depth_grid)
model_stack_all = stack_stepmodels(models, vpvs_values, np.arange(len(models)), depth_grid)
model_stats_best = summarize_curves(model_stack_best)
model_stats_all = summarize_curves(model_stack_all)

print(f"Computed PPC for {best_count} best models out of {len(misfits)} proposals.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

ax_model = axes[0]
ax_model.fill_between(depth_grid, model_stats_all["p5"], model_stats_all["p95"],
                      color="tab:blue", alpha=0.15, label="posterior p5-p95")
ax_model.fill_between(depth_grid, model_stats_all["q1"], model_stats_all["q3"],
                      color="tab:blue", alpha=0.3, label="posterior q1-q3")
ax_model.plot(depth_grid, model_stats_all["median"], color="tab:blue", label="posterior median")
ax_model.plot(depth_grid, model_stats_all["mean"], color="tab:blue", linestyle="--", label="posterior mean")
ax_model.plot(depth_grid, model_stats_best["median"], color="tab:red", label="best-100 median")
ax_model.plot(true_model_depth, true_model_vs, color="black", linewidth=2, label="true model")
ax_model.set_xlabel("Depth (km)")
ax_model.set_ylabel("Vs (km/s)")
ax_model.set_title("Model Posterior vs Best-100 Median")
ax_model.grid(True)
ax_model.legend()

ax_disp = axes[1]
ax_disp.plot(true_x, true_y, "k.", label="observed")
ax_disp.fill_between(true_x, disp_stats_all["p5"], disp_stats_all["p95"],
                    color="tab:orange", alpha=0.15, label="posterior p5-p95")
ax_disp.fill_between(true_x, disp_stats_all["q1"], disp_stats_all["q3"],
                    color="tab:orange", alpha=0.3, label="posterior q1-q3")
ax_disp.plot(true_x, disp_stats_all["median"], color="tab:orange", label="posterior median")
ax_disp.plot(true_x, disp_stats_all["mean"], color="tab:orange", linestyle="--", label="posterior mean")
ax_disp.plot(true_x, disp_stats_best["median"], color="tab:red", label="best-100 median")
ax_disp.set_xlabel("Period (s)")
ax_disp.set_ylabel("Group velocity (km/s)")
ax_disp.set_title("Dispersion Posterior vs Best-100 Median")
ax_disp.grid(True)
ax_disp.legend()

plt.tight_layout()
plt.show()


In [ ]:
def mse_rmse(observed, predicted, axis=None):
    diff = predicted - observed
    mse = np.nanmean(diff ** 2, axis=axis)
    rmse = np.sqrt(mse)
    return mse, rmse

def summarize_misfits(misfit_values):
    return {
        "mean": float(np.nanmean(misfit_values)),
        "median": float(np.nanmedian(misfit_values)),
        "q1": float(np.nanpercentile(misfit_values, 25)),
        "q3": float(np.nanpercentile(misfit_values, 75)),
        "p5": float(np.nanpercentile(misfit_values, 5)),
        "p95": float(np.nanpercentile(misfit_values, 95)),
    }


In [ ]:
# Misfits for the best-100 dispersion curves
best_mse, best_rmse = mse_rmse(true_y, best_curves, axis=1)
print("Best-100 dispersion MSE (mean):", np.nanmean(best_mse))
print("Best-100 dispersion RMSE (mean):", np.nanmean(best_rmse))


In [ ]:
# Misfit stats for all simulated dispersion curves
all_mse, all_rmse = mse_rmse(true_y, all_curve_ys, axis=1)

disp_mse_stats = summarize_misfits(all_mse)
disp_rmse_stats = summarize_misfits(all_rmse)

print("All-curve MSE stats:", disp_mse_stats)
print("All-curve RMSE stats:", disp_rmse_stats)


In [ ]:
# Misfit stats between true model and all proposed models
true_vs_interp = np.interp(depth_grid, true_model_depth, true_model_vs)
model_mse, model_rmse = mse_rmse(true_vs_interp, model_stack_all, axis=1)

model_mse_stats = summarize_misfits(model_mse)
model_rmse_stats = summarize_misfits(model_rmse)

print("Model MSE stats:", model_mse_stats)
print("Model RMSE stats:", model_rmse_stats)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(all_mse, bins=40, color="tab:blue", alpha=0.8)
axes[0].set_title("Dispersion MSE Histogram")
axes[0].set_xlabel("MSE")
axes[0].set_ylabel("Count")
axes[0].grid(True, alpha=0.3)

axes[1].hist(all_rmse, bins=40, color="tab:orange", alpha=0.8)
axes[1].set_title("Dispersion RMSE Histogram")
axes[1].set_xlabel("RMSE")
axes[1].set_ylabel("Count")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
